In [ ]:
import pandas as pd
from pathlib import Path

print("Python Environment is Working!")

In [ ]:
from pathlib import Path

DATASET_DIR = Path(r"C:\Users\siddh\my-dataset")

print("Dataset path:", DATASET_DIR)
print("Dataset exists:", DATASET_DIR.exists())

In [ ]:
for path in DATASET_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(r"C:\Users\siddh\my-dataset"))

In [ ]:
audio_path = DATASET_DIR / "audio" / "train" / "audio-train-00001.parquet"

audio_df = pd.read_parquet(audio_path)

print("Shape:", audio_df.shape)
print("\nColumns:")
print(audio_df.columns.tolist())

In [ ]:
print("Audio type:", type(audio_df.loc[0, "audio"]))
print("Audio value preview:", audio_df.loc[0, "audio"])
print("Audio_type:", audio_df.loc[0, "audio_type"])
print("Participant ID:", audio_df.loc[0, "participant_id"])

In [ ]:
print(audio_df["audio_type"].value_counts(dropna=False))

In [ ]:
print("Unique participants:", audio_df["participant_id"].nunique())
print("Total recordings:", len(audio_df))

print("\nRecordings per participant:")
print(audio_df["participant_id"].value_counts().head(10))

In [ ]:
metadata_path = DATASET_DIR / "metadata" / "train.parquet"

metadata_df = pd.read_parquet(metadata_path)

print("Shape:", metadata_df.shape)
print("\nColumns:")
print(metadata_df.columns.tolist())

In [ ]:
audio_ids = set(audio_df["participant_id"].astype(str))
metadata_ids = set(metadata_df["id"].astype(str))

matching_ids = audio_ids & metadata_ids

print("Audio participants:", len(audio_ids))
print("Metadata participants:", len(metadata_ids))
print("Matching participants:", len(matching_ids))

In [ ]:
target_columns = [
    "asthma",
    "pneumonia",
    "cough",
    "fever",
    "loss_of_smell",
]

for col in target_columns:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False))

In [ ]:
for col in ["asthma", "pneumonia", "cough", "fever", "loss_of_smell"]:
    print(f"\n--- {col} ---")
    print("Unique values:", metadata_df[col].unique())

In [ ]:
for col in [

    "covid_status",

    "test_status",

    "smoker",

    "others_resp",

    "others_preexist",

    "vacc",

]:

    print(f"\n--- {col} ---")

    print(metadata_df[col].value_counts(dropna=False))

In [ ]:
for col in ["smoker", "others_resp", "others_preexist", "vacc"]:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False).to_string())

In [ ]:
train_metadata = pd.read_parquet(
    DATASET_DIR / "metadata" / "train.parquet"
)

val_metadata = pd.read_parquet(
    DATASET_DIR / "metadata" / "val.parquet"
)

test_metadata = pd.read_parquet(
    DATASET_DIR / "metadata" / "test.parquet"
)

train_ids = set(train_metadata["id"].astype(str))
val_ids = set(val_metadata["id"].astype(str))
test_ids = set(test_metadata["id"].astype(str))

print("Train participants:", len(train_ids))
print("Validation participants:", len(val_ids))
print("Test participants:", len(test_ids))

print("\nTrain ∩ Validation:", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(val_ids & test_ids))

In [ ]:
candidate_targets = [
    "asthma",
    "pneumonia",
    "fever",
    "loss_of_smell",
    "covid_status",
]

for col in candidate_targets:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False).to_string())

In [ ]:
candidate_targets = [
    "asthma",
    "pneumonia",
    "fever",
    "loss_of_smell",
    "covid_status",
]

for col in candidate_targets:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False).to_string())

In [ ]:
print(metadata_df["covid_status"].value_counts(dropna=False).to_string())

In [ ]:
quality_columns = [
    "cough_heavy_quality",
    "cough_shallow_quality",
    "breathing_deep_quality",
    "breathing_shallow_quality",
]

for col in quality_columns:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False).to_string())

In [ ]:
cough_types = ["cough-heavy", "cough-shallow"]

cough_df = audio_df[audio_df["audio_type"].isin(cough_types)].copy()

print("Total cough recordings:", len(cough_df))

for audio_type in cough_types:
    subset = cough_df[cough_df["audio_type"] == audio_type].copy()

    quality_col = (
        "cough_heavy_quality"
        if audio_type == "cough-heavy"
        else "cough_shallow_quality"
    )

    subset = subset.merge(
        metadata_df[["id", quality_col]],
        left_on="participant_id",
        right_on="id",
        how="left"
    )

    print(f"\n--- {audio_type} ---")
    print("Total:", len(subset))
    print(
        subset[quality_col]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

In [ ]:
comparison_df = cough_df.merge(
    metadata_df[
        [
            "id",
            "cough_heavy_quality",
            "cough_shallow_quality",
        ]
    ],
    left_on="participant_id",
    right_on="id",
    how="left",
)

comparison_df["metadata_quality"] = comparison_df.apply(
    lambda row: (
        row["cough_heavy_quality"]
        if row["audio_type"] == "cough-heavy"
        else row["cough_shallow_quality"]
    ),
    axis=1,
)

print(
    comparison_df[
        [
            "participant_id",
            "audio_type",
            "quality_score",
            "metadata_quality",
        ]
    ].head(20).to_string(index=False)
)

print("\nQuality score combinations:")
print(
    comparison_df[
        ["quality_score", "metadata_quality"]
    ]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

In [ ]:
print(metadata_df[["a", "g"]].head(10).to_string())

In [ ]:
mlp_input_columns = ["a", "g", "smoker"]

for col in mlp_input_columns:
    print(f"\n--- {col} ---")
    print("Data type:", metadata_df[col].dtype)
    print("Missing:", metadata_df[col].isna().sum())
    print(metadata_df[col].value_counts(dropna=False).to_string())

In [ ]:
print("Gender data type:", metadata_df["g"].dtype)
print("Gender missing:", metadata_df["g"].isna().sum())
print("\nGender distribution:")
print(metadata_df["g"].value_counts(dropna=False).to_string())

In [ ]:
print("Smoker data type:", metadata_df["smoker"].dtype)
print("Smoker missing:", metadata_df["smoker"].isna().sum())
print("\nSmoker distribution:")
print(metadata_df["smoker"].value_counts(dropna=False).to_string())

In [ ]:
from pathlib import Path
import pandas as pd


COUGH_TYPES = ["cough-heavy", "cough-shallow"]


def normalize_smoker(value):

    if pd.isna(value):
        return None

    if value is True or value == "y":
        return 1

    if value is False or value == "n":
        return 0

    return None


def build_split(dataset_dir, split):

    dataset_dir = Path(dataset_dir)

    audio_dir = dataset_dir / "audio" / split
    metadata_path = dataset_dir / "metadata" / f"{split}.parquet"

    metadata_columns = [
        "id",
        "a",
        "g",
        "smoker",
        "asthma",
        "pneumonia",
        "fever",
        "loss_of_smell",
        "covid_status",
        "ihd",
    ]

    metadata = pd.read_parquet(
        metadata_path,
        columns=metadata_columns,
    )

    audio_files = sorted(audio_dir.glob("*.parquet"))

    if not audio_files:
        raise FileNotFoundError(
            f"No audio Parquet files found in: {audio_dir}"
        )

    audio_frames = []

    for audio_file in audio_files:

        frame = pd.read_parquet(
            audio_file,
            columns=[
                "participant_id",
                "audio_type",
                "audio",
                "quality_score",
            ],
        )

        frame = frame[
            frame["audio_type"].isin(COUGH_TYPES)
        ].copy()

        frame = frame[
            frame["quality_score"] >= 1
        ].copy()

        audio_frames.append(frame)

    audio = pd.concat(
        audio_frames,
        ignore_index=True,
    )

    result = audio.merge(
        metadata,
        left_on="participant_id",
        right_on="id",
        how="inner",
        validate="many_to_one",
    )

    result["smoker_clean"] = result["smoker"].apply(
        normalize_smoker
    )

    return result

In [ ]:
train_df = build_split(DATASET_DIR, "train")

print("Training rows:", len(train_df))

print("\nColumns:")
print(train_df.columns.tolist())

display(train_df.head())

In [ ]:
val_df = build_split(DATASET_DIR, "val")
test_df = build_split(DATASET_DIR, "test")

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nUnique participants:")
print("Train:", train_df["participant_id"].nunique())
print("Validation:", val_df["participant_id"].nunique())
print("Test:", test_df["participant_id"].nunique())

In [ ]:
TARGET_COLUMNS = [
    "asthma",
    "pneumonia",
    "fever",
    "loss_of_smell",
]

for df in [train_df, val_df, test_df]:
    for target in TARGET_COLUMNS:
        df[f"{target}_label"] = df[target].map(
            {True: 1}
        )

In [ ]:
for target in TARGET_COLUMNS:
    print(f"\n--- {target} ---")
    print(
        train_df[f"{target}_label"]
        .value_counts(dropna=False)
    )

In [ ]:
COVID_POSITIVE = {
    "positive_mild",
    "positive_moderate",
    "positive_asymp",
}

COVID_NEGATIVE = {
    "healthy",
    "no_resp_illness_exposed",
    "recovered_full",
    "resp_illness_not_identified",
}


def create_covid_label(value):
    if value in COVID_POSITIVE:
        return 1

    if value in COVID_NEGATIVE:
        return 0

    return float("nan")


for df in [train_df, val_df, test_df]:
    df["covid_label"] = df["covid_status"].apply(
        create_covid_label
    )

In [ ]:
print(
    train_df["covid_label"]
    .value_counts(dropna=False)
)

In [ ]:
import io

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf


sample_audio = train_df.iloc[0]["audio"]

audio_bytes = sample_audio["bytes"]

waveform, sample_rate = sf.read(io.BytesIO(audio_bytes))

print("Sample rate:", sample_rate)
print("Waveform shape:", waveform.shape)
print("Duration (seconds):", len(waveform) / sample_rate)

In [ ]:
mel_spectrogram = librosa.feature.melspectrogram(
    y=waveform.astype(np.float32),
    sr=sample_rate,
    n_mels=128,
    fmax=sample_rate // 2,
)

mel_db = librosa.power_to_db(
    mel_spectrogram,
    ref=np.max,
)

print("Mel spectrogram shape:", mel_db.shape)

In [ ]:
plt.figure(figsize=(10, 4))

librosa.display.specshow(
    mel_db,
    sr=sample_rate,
    x_axis="time",
    y_axis="mel",
)

plt.colorbar(format="%+2.0f dB")
plt.title("Cough Mel Spectrogram")
plt.tight_layout()
plt.show()

In [ ]:
import io

import librosa
import numpy as np
import soundfile as sf

def audio_to_melspectrogram(audio_dict, target_frames=256):

    audio_bytes = audio_dict["bytes"]

    waveform, sample_rate = sf.read(
        io.BytesIO(audio_bytes)
    )

    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)

    mel = librosa.feature.melspectrogram(
        y=waveform.astype(np.float32),
        sr=sample_rate,
        n_mels=128,
        fmax=sample_rate // 2,
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max,
    ).astype(np.float32)

    current_frames = mel_db.shape[1]

    if current_frames < target_frames:
        pad_width = target_frames - current_frames

        mel_db = np.pad(
            mel_db,
            ((0, 0), (0, pad_width)),
            mode="constant",
            constant_values=mel_db.min(),
        )

    elif current_frames > target_frames:
        mel_db = mel_db[:, :target_frames]

    return mel_db

In [ ]:
test_mel = audio_to_melspectrogram(
    train_df.iloc[0]["audio"]
)

print("Spectrogram shape:", test_mel.shape)
print("Data type:", test_mel.dtype)
print("Minimum:", test_mel.min())
print("Maximum:", test_mel.max())

In [ ]:
import torch
from torch.utils.data import Dataset


class CoughSenseDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        mel = audio_to_melspectrogram(row["audio"])

        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)

        age = float(row["a"])

        gender = row["g"]

        if gender == "male":
            gender_encoded = [1.0, 0.0, 0.0]
        elif gender == "female":
            gender_encoded = [0.0, 1.0, 0.0]
        else:
            gender_encoded = [0.0, 0.0, 1.0]

        smoker = row["smoker_clean"]

        if pd.isna(smoker):
            smoker_value = 0.0
            smoker_missing = 1.0
        else:
            smoker_value = float(smoker)
            smoker_missing = 0.0

        metadata_features = torch.tensor(
            [age / 100.0, *gender_encoded, smoker_value, smoker_missing],
            dtype=torch.float32,
        )

        
        screening_label = row["respiratory_concern"]

        if pd.isna(screening_label):
            screening_label = -1.0
        else:
            screening_label = float(screening_label)

        labels = torch.tensor(
            [screening_label],
            dtype=torch.float32,
)

        return {
            "spectrogram": mel,
            "metadata": metadata_features,
            "labels": labels,
        }

In [ ]:
train_dataset = CoughSenseDataset(train_df)

sample = train_dataset[0]

print("Spectrogram:", sample["spectrogram"].shape)
print("Metadata:", sample["metadata"].shape)
print("Labels:", sample["labels"])

In [ ]:
import torch
import torch.nn as nn


class CoughSenseModel(nn.Module):
    def __init__(self, metadata_dim=6, num_targets=1):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.audio_projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.metadata_mlp = nn.Sequential(
            nn.Linear(metadata_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 32),
            nn.ReLU(),
        )

        self.fusion = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.heads = nn.ModuleList([
            nn.Linear(64, 1) for _ in range(num_targets)
        ])

    def forward(self, spectrogram, metadata):
        audio_features = self.cnn(spectrogram)
        audio_features = self.audio_projection(audio_features)

        metadata_features = self.metadata_mlp(metadata)

        combined = torch.cat(
            [audio_features, metadata_features],
            dim=1,
        )

        fused = self.fusion(combined)

        outputs = torch.cat(
            [head(fused) for head in self.heads],
            dim=1,
        )

        return outputs

In [ ]:
model = CoughSenseModel()

sample_spectrogram = sample["spectrogram"].unsqueeze(0)
sample_metadata = sample["metadata"].unsqueeze(0)

output = model(
    sample_spectrogram,
    sample_metadata,
)

print("Output shape:", output.shape)
print("Output:", output)

In [ ]:
import torch
import torch.nn.functional as F


def masked_multitask_loss(outputs, targets):

    total_loss = 0.0
    valid_tasks = 0

    for task_index in range(outputs.shape[1]):
        task_outputs = outputs[:, task_index]
        task_targets = targets[:, task_index]

        mask = task_targets != -1

        if mask.any():
            loss = F.binary_cross_entropy_with_logits(
                task_outputs[mask],
                task_targets[mask],
            )

            total_loss += loss
            valid_tasks += 1

    if valid_tasks == 0:
        return outputs.sum() * 0.0

    return total_loss / valid_tasks

In [ ]:
loss = masked_multitask_loss(
    output,
    sample["labels"].unsqueeze(0),
)

print("Loss:", loss.item())

In [ ]:
from torch.utils.data import DataLoader

train_dataset = CoughSenseDataset(train_df)
val_dataset = CoughSenseDataset(val_df)
test_dataset = CoughSenseDataset(test_df)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

In [ ]:
batch = next(iter(train_loader))

print("Spectrogram:", batch["spectrogram"].shape)
print("Metadata:", batch["metadata"].shape)
print("Labels:", batch["labels"].shape)

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0

    for batch in loader:
        spectrograms = batch["spectrogram"].to(device)
        metadata = batch["metadata"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            spectrograms,
            metadata
        )

        loss = masked_multitask_loss(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate(model, loader, device):
    model.eval()

    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            spectrograms = batch["spectrogram"].to(device)
            metadata = batch["metadata"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                spectrograms,
                metadata
            )

            loss = masked_multitask_loss(
                outputs,
                labels
            )

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = CoughSenseModel(
    metadata_dim=6,
    num_targets=5,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

In [ ]:
train_loss = train_one_epoch(
    model,
    train_loader,
    optimizer,
    device,
)

val_loss = validate(
    model,
    val_loader,
    device,
)

print("Train loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
print("Missing audio:", train_df["audio"].isna().sum())
print("Total training rows:", len(train_df))

In [ ]:
print("Training rows before:", len(train_df))

train_df = train_df.dropna(subset=["audio"]).reset_index(drop=True)

print("Training rows after:", len(train_df))
print("Removed:", 3579 - len(train_df))

In [ ]:
print("Validation missing audio:", val_df["audio"].isna().sum())
print("Test missing audio:", test_df["audio"].isna().sum())

In [ ]:
print("Validation rows before:", len(val_df))

val_df = val_df.dropna(subset=["audio"]).reset_index(drop=True)

print("Validation rows after:", len(val_df))
print("Removed:", 1)

In [ ]:
train_dataset = CoughSenseDataset(train_df)
val_dataset = CoughSenseDataset(val_df)
test_dataset = CoughSenseDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

In [ ]:
batch = next(iter(train_loader))

print("Spectrogram:", batch["spectrogram"].shape)
print("Metadata:", batch["metadata"].shape)
print("Labels:", batch["labels"].shape)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = CoughSenseModel(
    metadata_dim=6,
    num_targets=5
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
train_loss = train_one_epoch(
    model,
    train_loader,
    optimizer,
    device
)

val_loss = validate(
    model,
    val_loader,
    device
)

print("Train loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
best_val_loss = float("inf")
best_model_state = None

num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    val_loss = validate(
        model,
        val_loader,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

In [ ]:
model.load_state_dict(best_model_state)
model.to(device)
model.eval()

In [ ]:
model.load_state_dict(best_model_state)
model.to(device)
model.eval()

print("Best model restored.")
print("Best validation loss:", best_val_loss)

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

TARGET_NAMES = [
    "Asthma",
    "Pneumonia",
    "Fever",
    "Loss of smell",
    "COVID-related",
]


def collect_test_predictions(model, loader, device):
    model.eval()

    all_outputs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            spectrograms = batch["spectrogram"].to(device)
            metadata = batch["metadata"].to(device)

            outputs = model(
                spectrograms,
                metadata,
            )

            all_outputs.append(outputs.cpu())
            all_labels.append(batch["labels"])

    outputs = torch.cat(all_outputs).numpy()
    labels = torch.cat(all_labels).numpy()

    return outputs, labels


test_outputs, test_labels = collect_test_predictions(
    model,
    test_loader,
    device,
)

print("Test output shape:", test_outputs.shape)
print("Test label shape:", test_labels.shape)

In [ ]:
test_probabilities = 1 / (1 + np.exp(-test_outputs))

for i, target_name in enumerate(TARGET_NAMES):
    y_true = test_labels[:, i]
    y_prob = test_probabilities[:, i]

    mask = y_true != -1

    y_true = y_true[mask].astype(int)
    y_prob = y_prob[mask]

    if len(np.unique(y_true)) < 2:
        print(f"\n--- {target_name} ---")
        print("Not enough classes for evaluation.")
        continue

    y_pred = (y_prob >= 0.5).astype(int)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    auc = roc_auc_score(
        y_true,
        y_prob,
    )

    print(f"\n--- {target_name} ---")
    print("Known test samples:", len(y_true))
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"ROC-AUC:   {auc:.4f}")

In [ ]:
SCREENING_POSITIVE_COVID = {
    "positive_mild",
    "positive_moderate",
    "positive_asymp",
}

SCREENING_NEGATIVE_COVID = {
    "healthy",
}


def create_respiratory_concern_label(row):

    symptom_columns = [
        "asthma",
        "pneumonia",
        "fever",
        "loss_of_smell",
    ]

    for column in symptom_columns:
        if row[column] is True:
            return 1

    if row["covid_status"] in SCREENING_POSITIVE_COVID:
        return 1

    if row["covid_status"] in SCREENING_NEGATIVE_COVID:
        return 0

    return float("nan")


for df in [train_df, val_df, test_df]:
    df["respiratory_concern"] = df.apply(
        create_respiratory_concern_label,
        axis=1,
    )

In [ ]:
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n--- {name} ---")
    print(
        df["respiratory_concern"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

In [ ]:
train_dataset = CoughSenseDataset(train_df)
val_dataset = CoughSenseDataset(val_df)
test_dataset = CoughSenseDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

In [ ]:
batch = next(iter(train_loader))

print("Spectrogram:", batch["spectrogram"].shape)
print("Metadata:", batch["metadata"].shape)
print("Labels:", batch["labels"].shape)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

screening_model = CoughSenseModel(
    metadata_dim=6,
    num_targets=1,
).to(device)

screening_optimizer = torch.optim.Adam(
    screening_model.parameters(),
    lr=1e-3,
)

In [ ]:
screening_train_loss = train_one_epoch(
    screening_model,
    train_loader,
    screening_optimizer,
    device,
)

screening_val_loss = validate(
    screening_model,
    val_loader,
    device,
)

print("Train loss:", screening_train_loss)
print("Validation loss:", screening_val_loss)

In [ ]:
best_screening_val_loss = float("inf")
best_screening_model_state = None

num_epochs = 10

for epoch in range(num_epochs):

    train_loss = train_one_epoch(
        screening_model,
        train_loader,
        screening_optimizer,
        device,
    )

    val_loss = validate(
        screening_model,
        val_loader,
        device,
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_screening_val_loss:
        best_screening_val_loss = val_loss

        best_screening_model_state = {
            key: value.detach().cpu().clone()
            for key, value in screening_model.state_dict().items()
        }

In [ ]:
screening_model.load_state_dict(
    best_screening_model_state
)

screening_model.to(device)
screening_model.eval()

print("Best screening model restored.")
print("Best validation loss:", best_screening_val_loss)

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

screening_model.eval()

all_outputs = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        spectrograms = batch["spectrogram"].to(device)
        metadata = batch["metadata"].to(device)

        outputs = screening_model(
            spectrograms,
            metadata,
        )

        all_outputs.append(outputs.cpu())
        all_labels.append(batch["labels"])

test_outputs = torch.cat(all_outputs).numpy().flatten()
test_labels = torch.cat(all_labels).numpy().flatten()

mask = test_labels != -1

y_true = test_labels[mask].astype(int)
y_prob = 1 / (1 + np.exp(-test_outputs[mask]))

y_pred = (y_prob >= 0.5).astype(int)

print("Known test samples:", len(y_true))
print("Positive samples:", (y_true == 1).sum())
print("Negative samples:", (y_true == 0).sum())

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0,
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0,
)

auc = roc_auc_score(
    y_true,
    y_prob,
)

print("\n--- CoughSense Respiratory Screening ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
import os

os.makedirs("models", exist_ok=True)

screening_model.load_state_dict(
    best_screening_model_state
)

torch.save(
    screening_model.state_dict(),
    "models/coughsense_screening_model.pth"
)

print("Model saved successfully.")

In [ ]:
import os

print(
    "Model exists:",
    os.path.exists("models/coughsense_screening_model.pth")
)

print(
    "Model size:",
    os.path.getsize(
        "models/coughsense_screening_model.pth"
    ) / (1024 * 1024),
    "MB"
)

In [ ]:
import os

os.makedirs("models", exist_ok=True)

torch.save(
    screening_model.state_dict(),
    "models/coughsense_screening_model.pth"
)

print(
    os.path.exists(
        "models/coughsense_screening_model.pth"
    )
)

In [ ]:
import os
import torch

os.makedirs("../models", exist_ok=True)

torch.save(
    screening_model.state_dict(),
    "../models/coughsense_screening_model.pth"
)

print("Saved:", os.path.exists(
    "../models/coughsense_screening_model.pth"
))